# MHR-SafeGuard — Polished Research Notebook v3 — Long-Term Framework

Google Colab notebook for the Kaggle dataset:

`MyDrive/Datasets/MentalHealth/anxiety_depression_data.csv`

All outputs are saved progressively in:

`MyDrive/Outputs/MHR_SafeGuard_Research/`

This upgraded version includes: leakage-aware modeling, nested cross-validation, optional Optuna refinement, repeated cross-validation with statistical comparison, bootstrap confidence intervals, split-conformal prediction intervals, calibration diagnostics, permutation importance, SHAP/LIME, robust DiCE with automatic retries and fallback what-if analysis, uncertainty-aware safety policy engine, missing-data robustness, subgroup/fairness diagnostics, target-weight sensitivity, environment capture, manuscript-ready exports, HTML/Markdown/Word reports, and progressive logging.


This version adds configuration artifacts, optional experiment tracking, external-validation hooks, distribution-shift diagnostics, model/data cards, and inference deployment assets.

In [ ]:
import os, sys, subprocess, warnings, json, random, math, joblib
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive')
else:
    ROOT = Path.cwd()

PROJECT_NAME = "MHR_SafeGuard_Research"
BASE_DIR = ROOT / "Outputs" / PROJECT_NAME
FIG_DIR = BASE_DIR / "Figures"
TABLE_DIR = BASE_DIR / "Tables"
MODEL_DIR = BASE_DIR / "Models"
REPORT_DIR = BASE_DIR / "Reports"
LOG_PATH = BASE_DIR / "Outputs_summary.txt"
for d in [BASE_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
LOG_PATH.write_text("", encoding="utf-8")

def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def log(msg):
    line = f"[{now()}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def section(title):
    print("\n" + "="*90)
    print(title)
    print("="*90)
    log(title)

section("Notebook initialized")
print("Output directory:", BASE_DIR)


In [ ]:
section("Installing optional packages")
# The notebook works even if some optional packages fail to install.
if IN_COLAB:
    optional_packages = [
        ("shap", "shap"),
        ("lime", "lime"),
        ("dice-ml", "dice_ml"),
        ("lightgbm", "lightgbm"),
        ("xgboost", "xgboost"),
        ("optuna", "optuna"),
        ("plotly", "plotly"),
        ("python-docx", "docx"),
    ]
    for pkg, import_name in optional_packages:
        try:
            __import__(import_name)
            log(f"Available: {pkg}")
        except Exception:
            log(f"Installing: {pkg}")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
else:
    log("Non-Colab environment detected; package installation skipped.")


In [ ]:
section("Imports and helper functions")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import randint, uniform, loguniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, brier_score_loss
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.utils import check_random_state

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def metrics_dict(y_true, y_pred, prefix=""):
    return {
        f"{prefix}mae": float(mean_absolute_error(y_true, y_pred)),
        f"{prefix}rmse": rmse(y_true, y_pred),
        f"{prefix}r2": float(r2_score(y_true, y_pred)),
    }

def neg_rmse_scorer(estimator, X, y_true):
    return -rmse(y_true, estimator.predict(X))

def save_table(df, name):
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    log(f"Saved table: {path}")
    return path

def save_json(obj, name):
    path = TABLE_DIR / name
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    log(f"Saved JSON: {path}")
    return path

def save_fig(name, dpi=220):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.show()
    log(f"Saved figure: {path}")
    return path

def safe_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


In [ ]:
section("Environment capture")

import platform
import pkgutil

version_info = {
    "generated_at": now(),
    "python": sys.version,
    "platform": platform.platform(),
}
for module_name in ["numpy", "pandas", "sklearn", "scipy", "matplotlib", "shap", "lime", "dice_ml", "lightgbm", "xgboost", "optuna", "plotly"]:
    try:
        mod = __import__(module_name)
        version_info[module_name] = getattr(mod, "__version__", "available")
    except Exception:
        version_info[module_name] = "not_available"

save_json(version_info, "environment_versions.json")
try:
    freeze_path = REPORT_DIR / "pip_freeze.txt"
    result = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=False)
    freeze_path.write_text(result.stdout, encoding="utf-8")
    log(f"Saved environment package list: {freeze_path}")
except Exception as e:
    log(f"pip freeze capture skipped: {repr(e)}")


In [ ]:
section("Loading dataset")

DATA_PATH = ROOT / "Datasets" / "MentalHealth" / "anxiety_depression_data.csv"
fallbacks = [Path("/mnt/data/anxiety_depression_data.csv"), Path("anxiety_depression_data.csv")]
if DATA_PATH.exists():
    dataset_path = DATA_PATH
else:
    dataset_path = next((p for p in fallbacks if p.exists()), None)

if dataset_path is None:
    raise FileNotFoundError("Dataset not found. Expected MyDrive/Datasets/MentalHealth/anxiety_depression_data.csv")

df_raw = pd.read_csv(dataset_path)
log(f"Loaded dataset from {dataset_path}")
log(f"Dataset shape: {df_raw.shape}")
display(df_raw.head())
save_table(pd.DataFrame({"column": df_raw.columns, "dtype": df_raw.dtypes.astype(str).values}), "dataset_schema.csv")


In [ ]:
section("Constructing MHR target and leakage-aware features")

required = ["Depression_Score", "Anxiety_Score", "Stress_Level"]
for c in required:
    if c not in df_raw.columns:
        raise ValueError(f"Missing required column: {c}")

WEIGHTS = {"Depression_Score": 0.40, "Anxiety_Score": 0.30, "Stress_Level": 0.30}
df = df_raw.copy()
df["MHR"] = sum(WEIGHTS[c] * pd.to_numeric(df[c], errors="coerce") for c in WEIGHTS)

TARGET = "MHR"
ALL_FEATURES = [c for c in df.columns if c != TARGET]
LEAKAGE_COMPONENTS = list(WEIGHTS.keys())
SAFE_FEATURES = [c for c in ALL_FEATURES if c not in LEAKAGE_COMPONENTS]

target_summary = {
    "target": TARGET,
    "formula": "0.40*Depression_Score + 0.30*Anxiety_Score + 0.30*Stress_Level",
    "all_features": len(ALL_FEATURES),
    "leakage_aware_features": len(SAFE_FEATURES),
    "excluded_target_components": LEAKAGE_COMPONENTS,
}
save_json(target_summary, "target_and_features.json")
print(json.dumps(target_summary, indent=2))

plt.figure(figsize=(7,4))
plt.hist(df[TARGET].dropna(), bins=30)
plt.xlabel("MHR")
plt.ylabel("Frequency")
plt.title("Composite Mental Health Risk target")
save_fig("mhr_distribution.png")


In [ ]:
section("Train/test split and leakage diagnostics")

X_all = df[ALL_FEATURES].copy()
X_safe = df[SAFE_FEATURES].copy()
y = df[TARGET].astype(float)

y_bins = pd.qcut(y, q=5, labels=False, duplicates="drop")
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y_bins
)
X_train_safe = X_train_all[SAFE_FEATURES].copy()
X_test_safe = X_test_all[SAFE_FEATURES].copy()

ks_rows = []
for col in ALL_FEATURES:
    if pd.api.types.is_numeric_dtype(df[col]):
        a = pd.to_numeric(X_train_all[col], errors="coerce").dropna()
        b = pd.to_numeric(X_test_all[col], errors="coerce").dropna()
        if len(a) > 2 and len(b) > 2:
            s, p = stats.ks_2samp(a, b)
            ks_rows.append({
                "feature": col,
                "ks_statistic": float(s),
                "p_value": float(p),
                "train_mean": float(a.mean()),
                "test_mean": float(b.mean()),
                "mean_difference": float(abs(a.mean() - b.mean())),
            })
ks_df = pd.DataFrame(ks_rows).sort_values("ks_statistic")
save_table(ks_df, "ks_train_test_distribution_diagnostics.csv")
display(ks_df)

leakage_note = pd.DataFrame({
    "setting": ["full_features", "leakage_aware_features"],
    "target_components_in_predictors": [", ".join(LEAKAGE_COMPONENTS), "None"],
    "interpretation": [
        "Optimistic upper bound; includes the variables used to construct MHR.",
        "Preferred scientific setting; removes Depression, Anxiety, and Stress from predictors.",
    ],
})
save_table(leakage_note, "leakage_note.csv")
display(leakage_note)


In [ ]:
section("Preprocessing pipelines")

def infer_types(X):
    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat = [c for c in X.columns if c not in num]
    return num, cat

def make_preprocessor(X):
    num, cat = infer_types(X)
    transformers = []
    if num:
        transformers.append(("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num))
    if cat:
        transformers.append(("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", safe_onehot_encoder())
        ]), cat))
    return ColumnTransformer(transformers), num, cat

pre_all, num_all, cat_all = make_preprocessor(X_train_all)
pre_safe, num_safe, cat_safe = make_preprocessor(X_train_safe)

feature_summary = pd.DataFrame({
    "setting": ["full_features", "leakage_aware_features"],
    "numeric_count": [len(num_all), len(num_safe)],
    "categorical_count": [len(cat_all), len(cat_safe)],
    "numeric_features": [", ".join(num_all), ", ".join(num_safe)],
    "categorical_features": [", ".join(cat_all), ", ".join(cat_safe)],
})
save_table(feature_summary, "feature_type_summary.csv")
display(feature_summary)


In [ ]:
section("Model comparison with nested cross-validation")

models = {
    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "RandomForest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    "ExtraTrees": ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    "Ridge": Ridge(),
    "ElasticNet": ElasticNet(random_state=RANDOM_STATE, max_iter=10000),
    "SVR": SVR(),
}

try:
    from lightgbm import LGBMRegressor
    models["LightGBM"] = LGBMRegressor(random_state=RANDOM_STATE, verbosity=-1)
except Exception as e:
    log(f"LightGBM skipped: {repr(e)}")

try:
    from xgboost import XGBRegressor
    models["XGBoost"] = XGBRegressor(
        random_state=RANDOM_STATE, objective="reg:squarederror",
        n_estimators=120, verbosity=0, n_jobs=-1
    )
except Exception as e:
    log(f"XGBoost skipped: {repr(e)}")

param_spaces = {
    "GradientBoosting": {"model__n_estimators": randint(80, 160), "model__learning_rate": loguniform(0.02, 0.15), "model__max_depth": randint(2, 5)},
    "RandomForest": {"model__n_estimators": randint(80, 180), "model__max_depth": [None, 4, 6, 8, 12], "model__min_samples_leaf": randint(1, 6)},
    "ExtraTrees": {"model__n_estimators": randint(80, 180), "model__max_depth": [None, 4, 6, 8, 12], "model__min_samples_leaf": randint(1, 6)},
    "Ridge": {"model__alpha": loguniform(1e-3, 100)},
    "ElasticNet": {"model__alpha": loguniform(1e-4, 10), "model__l1_ratio": uniform(0.05, 0.90)},
    "SVR": {"model__C": loguniform(0.1, 100), "model__epsilon": loguniform(0.01, 0.5), "model__gamma": ["scale", "auto"]},
    "LightGBM": {"model__n_estimators": randint(80, 180), "model__learning_rate": loguniform(0.02, 0.15), "model__num_leaves": randint(15, 45)},
    "XGBoost": {"model__n_estimators": randint(80, 160), "model__learning_rate": loguniform(0.02, 0.15), "model__max_depth": randint(2, 5)},
}

def evaluate_setting(setting_name, X_train, X_test, preprocessor, outer_splits=3, inner_splits=3, n_iter=6):
    rows, fitted = [], {}
    outer_cv = KFold(n_splits=outer_splits, shuffle=True, random_state=RANDOM_STATE)
    for model_name, model in models.items():
        log(f"Evaluating {setting_name} | {model_name}")
        try:
            pipe = Pipeline([("pre", clone(preprocessor)), ("model", clone(model))])
            search = RandomizedSearchCV(
                estimator=pipe,
                param_distributions=param_spaces.get(model_name, {}),
                n_iter=n_iter if param_spaces.get(model_name, {}) else 1,
                scoring=neg_rmse_scorer,
                cv=inner_splits,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                refit=True,
                error_score="raise"
            )
            outer_scores = []
            for fold, (tr, va) in enumerate(outer_cv.split(X_train), start=1):
                X_tr, X_va = X_train.iloc[tr], X_train.iloc[va]
                y_tr, y_va = y_train.iloc[tr], y_train.iloc[va]
                fs = clone(search)
                fs.fit(X_tr, y_tr)
                fold_rmse = rmse(y_va, fs.predict(X_va))
                outer_scores.append(fold_rmse)
                log(f"{setting_name} | {model_name} | fold {fold} RMSE={fold_rmse:.4f}")
            search.fit(X_train, y_train)
            best_est = search.best_estimator_
            fitted[(setting_name, model_name)] = best_est
            pred_tr = best_est.predict(X_train)
            pred_te = best_est.predict(X_test)
            row = {
                "setting": setting_name,
                "model": model_name,
                "nested_cv_rmse_mean": float(np.mean(outer_scores)),
                "nested_cv_rmse_std": float(np.std(outer_scores)),
                "best_params": json.dumps(search.best_params_),
            }
            row.update(metrics_dict(y_train, pred_tr, "train_"))
            row.update(metrics_dict(y_test, pred_te, "test_"))
            rows.append(row)
        except Exception as e:
            log(f"Failed {setting_name} | {model_name}: {repr(e)}")
            rows.append({"setting": setting_name, "model": model_name, "error": repr(e)})
    return pd.DataFrame(rows), fitted

res_full, fit_full = evaluate_setting("full_features", X_train_all, X_test_all, pre_all)
res_safe, fit_safe = evaluate_setting("leakage_aware_features", X_train_safe, X_test_safe, pre_safe)

results = pd.concat([res_full, res_safe], ignore_index=True)
valid_results = results.dropna(subset=["test_rmse"]).sort_values(["setting", "test_rmse"])
save_table(results, "model_comparison_all_attempts.csv")
save_table(valid_results, "model_comparison_valid.csv")
display(valid_results)

plt.figure(figsize=(10,5))
plot_df = valid_results.copy()
plt.barh(plot_df["setting"] + " | " + plot_df["model"], plot_df["test_rmse"])
plt.xlabel("Holdout RMSE")
plt.title("Model comparison")
plt.gca().invert_yaxis()
save_fig("model_comparison_rmse.png")


In [ ]:
section("Repeated cross-validation and statistical comparison")

from sklearn.model_selection import RepeatedKFold
from scipy.stats import friedmanchisquare, wilcoxon

REPEATED_CV_SPLITS = 5
REPEATED_CV_REPEATS = 3
MAX_MODELS_FOR_REPEATED_CV = 5

rep_rows = []
stats_rows = []
cv_results_by_model = {}

safe_models = valid_results[valid_results["setting"] == "leakage_aware_features"].sort_values("test_rmse")
selected_models_for_rep = safe_models["model"].head(MAX_MODELS_FOR_REPEATED_CV).tolist()
log(f"Repeated CV models: {selected_models_for_rep}")

rkf = RepeatedKFold(n_splits=REPEATED_CV_SPLITS, n_repeats=REPEATED_CV_REPEATS, random_state=RANDOM_STATE)
for model_name in selected_models_for_rep:
    est = clone(fit_safe[("leakage_aware_features", model_name)])
    fold_rmses = []
    for fold_id, (tr, va) in enumerate(rkf.split(X_train_safe), start=1):
        est_fold = clone(est)
        est_fold.fit(X_train_safe.iloc[tr], y_train.iloc[tr])
        pred_va = est_fold.predict(X_train_safe.iloc[va])
        fold_rmse = rmse(y_train.iloc[va], pred_va)
        fold_rmses.append(fold_rmse)
        rep_rows.append({"model": model_name, "fold_id": fold_id, "rmse": fold_rmse})
    cv_results_by_model[model_name] = np.asarray(fold_rmses, dtype=float)

rep_cv_df = pd.DataFrame(rep_rows)
save_table(rep_cv_df, "repeated_cv_fold_scores.csv")

if not rep_cv_df.empty:
    rep_summary = rep_cv_df.groupby("model")["rmse"].agg(["count", "mean", "std", "median", "min", "max"]).reset_index().sort_values("mean")
    save_table(rep_summary, "repeated_cv_summary.csv")
    display(rep_summary)

    plt.figure(figsize=(9, 5))
    data = [cv_results_by_model[m] for m in rep_summary["model"]]
    plt.boxplot(data, labels=rep_summary["model"], vert=True)
    plt.ylabel("Repeated-CV RMSE")
    plt.title("Repeated cross-validation RMSE distribution")
    plt.xticks(rotation=30, ha="right")
    save_fig("repeated_cv_rmse_boxplot.png")

    if len(cv_results_by_model) >= 3:
        ordered_models = list(rep_summary["model"])
        stat, p = friedmanchisquare(*[cv_results_by_model[m] for m in ordered_models])
        stats_rows.append({"test": "Friedman", "models": ", ".join(ordered_models), "statistic": float(stat), "p_value": float(p)})

    best_rep_model = rep_summary.iloc[0]["model"]
    for model_name in rep_summary["model"]:
        if model_name == best_rep_model:
            continue
        try:
            stat, p = wilcoxon(cv_results_by_model[best_rep_model], cv_results_by_model[model_name])
            stats_rows.append({"test": "Wilcoxon_vs_best", "best_model": best_rep_model, "comparison_model": model_name, "statistic": float(stat), "p_value": float(p)})
        except Exception as e:
            stats_rows.append({"test": "Wilcoxon_vs_best", "best_model": best_rep_model, "comparison_model": model_name, "error": repr(e)})

stats_df = pd.DataFrame(stats_rows)
if not stats_df.empty:
    save_table(stats_df, "repeated_cv_statistical_tests.csv")
    display(stats_df)
else:
    log("Statistical comparison skipped because repeated-CV results were insufficient.")


In [ ]:
section("Final leakage-aware model selection")

safe_valid = valid_results[valid_results["setting"] == "leakage_aware_features"].copy()
if safe_valid.empty:
    raise RuntimeError("No valid leakage-aware model was fitted. Inspect model_comparison_all_attempts.csv")

best_row = safe_valid.sort_values("test_rmse").iloc[0]
BEST_MODEL_NAME = best_row["model"]
final_model = fit_safe[("leakage_aware_features", BEST_MODEL_NAME)]

pred_train = final_model.predict(X_train_safe)
pred_test = final_model.predict(X_test_safe)

final_metrics = {"best_model": BEST_MODEL_NAME, "feature_setting": "leakage_aware_features"}
final_metrics.update(metrics_dict(y_train, pred_train, "train_"))
final_metrics.update(metrics_dict(y_test, pred_test, "test_"))
save_json(final_metrics, "final_model_metrics.json")

model_path = MODEL_DIR / "final_leakage_aware_model.joblib"
joblib.dump(final_model, model_path)
log(f"Saved final model: {model_path}")
print(json.dumps(final_metrics, indent=2))

pred_df = X_test_safe.copy()
pred_df["Actual_MHR"] = y_test.values
pred_df["Predicted_MHR"] = pred_test
pred_df["Residual"] = y_test.values - pred_test
save_table(pred_df, "holdout_predictions.csv")

plt.figure(figsize=(5.5,5.5))
plt.scatter(y_test, pred_test, alpha=0.75)
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual MHR")
plt.ylabel("Predicted MHR")
plt.title(f"Actual vs predicted — {BEST_MODEL_NAME}")
save_fig("final_actual_vs_predicted.png")

plt.figure(figsize=(7,4))
plt.hist(y_test.values - pred_test, bins=30)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Residual distribution")
save_fig("final_residual_distribution.png")


In [ ]:
section("Optional Optuna refinement of the selected leakage-aware model")

RUN_OPTUNA_REFINEMENT = True
OPTUNA_TRIALS = 20
OPTUNA_CV_SPLITS = 3

optuna_summary = {"enabled": RUN_OPTUNA_REFINEMENT, "status": "not_run"}

try:
    import optuna
    if not RUN_OPTUNA_REFINEMENT:
        raise RuntimeError("Optuna refinement disabled by RUN_OPTUNA_REFINEMENT=False")

    def build_optuna_estimator(trial, model_name):
        if model_name == "GradientBoosting":
            model = GradientBoostingRegressor(
                random_state=RANDOM_STATE,
                n_estimators=trial.suggest_int("n_estimators", 80, 260),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
                max_depth=trial.suggest_int("max_depth", 2, 5),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 8),
                subsample=trial.suggest_float("subsample", 0.65, 1.0),
            )
        elif model_name == "RandomForest":
            model = RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1,
                n_estimators=trial.suggest_int("n_estimators", 100, 320),
                max_depth=trial.suggest_categorical("max_depth", [None, 4, 6, 8, 12, 16]),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 8),
                max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", 1.0]),
            )
        elif model_name == "ExtraTrees":
            model = ExtraTreesRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1,
                n_estimators=trial.suggest_int("n_estimators", 100, 320),
                max_depth=trial.suggest_categorical("max_depth", [None, 4, 6, 8, 12, 16]),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 8),
                max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", 1.0]),
            )
        elif model_name == "Ridge":
            model = Ridge(alpha=trial.suggest_float("alpha", 1e-4, 200.0, log=True))
        elif model_name == "ElasticNet":
            model = ElasticNet(
                random_state=RANDOM_STATE,
                max_iter=20000,
                alpha=trial.suggest_float("alpha", 1e-5, 20.0, log=True),
                l1_ratio=trial.suggest_float("l1_ratio", 0.01, 0.99),
            )
        else:
            raise RuntimeError(f"No Optuna search space defined for {model_name}")
        return Pipeline([("pre", clone(pre_safe)), ("model", model)])

    def objective(trial):
        estimator = build_optuna_estimator(trial, BEST_MODEL_NAME)
        cv = KFold(n_splits=OPTUNA_CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        scores = []
        for tr, va in cv.split(X_train_safe):
            est = clone(estimator)
            est.fit(X_train_safe.iloc[tr], y_train.iloc[tr])
            scores.append(rmse(y_train.iloc[va], est.predict(X_train_safe.iloc[va])))
        return float(np.mean(scores))

    if BEST_MODEL_NAME in {"GradientBoosting", "RandomForest", "ExtraTrees", "Ridge", "ElasticNet"}:
        study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)
        optuna_model = build_optuna_estimator(study.best_trial, BEST_MODEL_NAME)
        optuna_model.fit(X_train_safe, y_train)
        optuna_pred_test = optuna_model.predict(X_test_safe)
        optuna_test_rmse = rmse(y_test, optuna_pred_test)
        previous_test_rmse = final_metrics["test_rmse"]
        optuna_summary = {
            "enabled": True,
            "status": "completed",
            "model": BEST_MODEL_NAME,
            "best_cv_rmse": float(study.best_value),
            "holdout_rmse": float(optuna_test_rmse),
            "previous_holdout_rmse": float(previous_test_rmse),
            "accepted": bool(optuna_test_rmse <= previous_test_rmse),
            "best_params": study.best_params,
        }
        save_json(optuna_summary, "optuna_refinement_summary.json")
        trials_df = study.trials_dataframe()
        save_table(trials_df, "optuna_trials.csv")
        if optuna_test_rmse <= previous_test_rmse:
            final_model = optuna_model
            pred_train = final_model.predict(X_train_safe)
            pred_test = final_model.predict(X_test_safe)
            final_metrics = {}
            final_metrics.update(metrics_dict(y_train, pred_train, "train_"))
            final_metrics.update(metrics_dict(y_test, pred_test, "test_"))
            final_metrics["model"] = BEST_MODEL_NAME
            final_metrics["optimization"] = "Optuna-refined"
            save_json(final_metrics, "final_model_metrics.json")
            joblib.dump(final_model, MODEL_DIR / "final_leakage_aware_model.joblib")
            log("Optuna-refined model accepted and saved.")
        else:
            log("Optuna-refined model not accepted because holdout RMSE did not improve.")
    else:
        optuna_summary = {"enabled": True, "status": "skipped", "reason": f"No Optuna search space for {BEST_MODEL_NAME}"}
        save_json(optuna_summary, "optuna_refinement_summary.json")
except Exception as e:
    optuna_summary = {"enabled": RUN_OPTUNA_REFINEMENT, "status": "skipped_or_failed", "reason": repr(e)}
    save_json(optuna_summary, "optuna_refinement_summary.json")
    log(f"Optuna refinement skipped or failed: {repr(e)}")


In [ ]:
section("Bootstrap confidence intervals")

rng = check_random_state(RANDOM_STATE)
B = 500
boot = []
yt, yp = y_test.values, np.asarray(pred_test)
for b in range(B):
    idx = rng.choice(np.arange(len(yt)), size=len(yt), replace=True)
    boot.append({"bootstrap_id": b, "mae": mean_absolute_error(yt[idx], yp[idx]), "rmse": rmse(yt[idx], yp[idx]), "r2": r2_score(yt[idx], yp[idx])})
boot_df = pd.DataFrame(boot)
ci_df = boot_df[["mae", "rmse", "r2"]].quantile([0.025, 0.5, 0.975]).T.reset_index()
ci_df.columns = ["metric", "ci_2_5", "median", "ci_97_5"]
save_table(boot_df, "bootstrap_metric_samples.csv")
save_table(ci_df, "bootstrap_metric_confidence_intervals.csv")
display(ci_df)


In [ ]:
section("Split-conformal prediction intervals")

X_tr_c, X_cal_c, y_tr_c, y_cal_c = train_test_split(X_train_safe, y_train, test_size=0.25, random_state=RANDOM_STATE)
conf_model = clone(final_model)
conf_model.fit(X_tr_c, y_tr_c)
cal_resid = np.abs(y_cal_c.values - conf_model.predict(X_cal_c))
alpha = 0.10
q_level = np.ceil((len(cal_resid) + 1) * (1 - alpha)) / len(cal_resid)
qhat = float(np.quantile(cal_resid, q_level, method="higher"))
conf_pred = conf_model.predict(X_test_safe)
lower, upper = conf_pred - qhat, conf_pred + qhat
coverage = float(np.mean((y_test.values >= lower) & (y_test.values <= upper)))
conformal_summary = {"alpha": alpha, "nominal_coverage": 1-alpha, "empirical_coverage": coverage, "qhat": qhat, "average_width": float(np.mean(upper-lower))}
save_json(conformal_summary, "conformal_summary.json")

conf_df = pd.DataFrame({"Actual_MHR": y_test.values, "Predicted_MHR": conf_pred, "Lower_90PI": lower, "Upper_90PI": upper, "Covered": (y_test.values >= lower) & (y_test.values <= upper)})
save_table(conf_df, "conformal_prediction_intervals.csv")
print(json.dumps(conformal_summary, indent=2))

plt.figure(figsize=(10,4))
n = min(60, len(conf_df))
order = np.argsort(conf_df["Predicted_MHR"].values)[:n]
plt.errorbar(np.arange(n), conf_df.iloc[order]["Predicted_MHR"], yerr=qhat, fmt="o", capsize=2)
plt.scatter(np.arange(n), conf_df.iloc[order]["Actual_MHR"], marker="x")
plt.xlabel("Sorted test instance")
plt.ylabel("MHR")
plt.title("90% split-conformal intervals")
save_fig("conformal_intervals_sample.png")


In [ ]:
section("Calibration for high-risk flag")

high_threshold = float(y_train.quantile(0.67))
low_threshold = float(y_train.quantile(0.33))

def normalize_score(scores, low=None, high=None):
    scores = np.asarray(scores, dtype=float)
    if low is None: low = np.percentile(scores, 1)
    if high is None: high = np.percentile(scores, 99)
    return np.clip((scores - low) / max(high-low, 1e-9), 0, 1)

high_true = (y_test.values >= high_threshold).astype(int)
high_prob = normalize_score(pred_test, y_train.quantile(0.05), y_train.quantile(0.95))
brier = float(brier_score_loss(high_true, high_prob))

bins = np.linspace(0,1,6)
rows = []
for i in range(len(bins)-1):
    if i < len(bins)-2:
        mask = (high_prob >= bins[i]) & (high_prob < bins[i+1])
    else:
        mask = (high_prob >= bins[i]) & (high_prob <= bins[i+1])
    if mask.sum() > 0:
        rows.append({"bin_low": bins[i], "bin_high": bins[i+1], "n": int(mask.sum()), "mean_predicted_probability": float(high_prob[mask].mean()), "observed_high_risk_rate": float(high_true[mask].mean())})
cal_df = pd.DataFrame(rows)
ece = float(np.sum(cal_df["n"] * np.abs(cal_df["mean_predicted_probability"] - cal_df["observed_high_risk_rate"])) / len(high_true))
cal_summary = {"high_threshold": high_threshold, "low_threshold": low_threshold, "brier_score": brier, "ece_5_bins": ece}
save_json(cal_summary, "calibration_summary.json")
save_table(cal_df, "calibration_bins.csv")
display(cal_df)

plt.figure(figsize=(5.5,5.5))
plt.plot([0,1],[0,1], linestyle="--")
plt.scatter(cal_df["mean_predicted_probability"], cal_df["observed_high_risk_rate"], s=80)
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed high-risk rate")
plt.title("High-risk calibration")
save_fig("high_risk_calibration.png")


In [ ]:
section("Global and local explanations")

perm = permutation_importance(final_model, X_test_safe, y_test, n_repeats=20, random_state=RANDOM_STATE, scoring=neg_rmse_scorer, n_jobs=-1)
perm_df = pd.DataFrame({"feature": SAFE_FEATURES, "importance_mean": perm.importances_mean, "importance_std": perm.importances_std}).sort_values("importance_mean", ascending=False)
save_table(perm_df, "permutation_importance.csv")
display(perm_df.head(15))

plt.figure(figsize=(8,5))
top = perm_df.head(15).iloc[::-1]
plt.barh(top["feature"], top["importance_mean"])
plt.xlabel("Permutation importance")
plt.title("Top leakage-aware predictors")
save_fig("permutation_importance_top15.png")

try:
    import shap
    pre = final_model.named_steps["pre"]
    core = final_model.named_steps["model"]
    Xtr = pre.transform(X_train_safe)
    Xte = pre.transform(X_test_safe)
    try:
        names = pre.get_feature_names_out()
    except Exception:
        names = [f"feature_{i}" for i in range(Xte.shape[1])]
    sample = Xte[:min(150, Xte.shape[0])]
    try:
        explainer = shap.TreeExplainer(core)
        vals = explainer.shap_values(sample)
    except Exception:
        background = shap.sample(Xtr, min(100, Xtr.shape[0]), random_state=RANDOM_STATE)
        explainer = shap.Explainer(core.predict, background)
        vals = explainer(sample).values
    shap_df = pd.DataFrame({"feature": names, "mean_abs_shap": np.mean(np.abs(vals), axis=0)}).sort_values("mean_abs_shap", ascending=False)
    save_table(shap_df, "shap_global_importance.csv")
    plt.figure(figsize=(8,5))
    top = shap_df.head(15).iloc[::-1]
    plt.barh(top["feature"], top["mean_abs_shap"])
    plt.xlabel("Mean |SHAP|")
    plt.title("SHAP global importance")
    save_fig("shap_global_importance_top15.png")
except Exception as e:
    log(f"SHAP skipped: {repr(e)}")

try:
    from lime.lime_tabular import LimeTabularExplainer
    pre = final_model.named_steps["pre"]
    core = final_model.named_steps["model"]
    Xtr = pre.transform(X_train_safe)
    Xte = pre.transform(X_test_safe)
    try:
        names = list(pre.get_feature_names_out())
    except Exception:
        names = [f"feature_{i}" for i in range(Xte.shape[1])]
    explainer = LimeTabularExplainer(np.asarray(Xtr), feature_names=names, mode="regression", discretize_continuous=True, random_state=RANDOM_STATE)
    pred_series = pd.Series(pred_test, index=X_test_safe.index)
    reps = {"low": pred_series.sort_values().index[len(pred_series)//10], "medium": (pred_series-pred_series.median()).abs().idxmin(), "high": pred_series.sort_values().index[-max(1, len(pred_series)//10)]}
    lime_rows = []
    for label, idx in reps.items():
        pos = list(X_test_safe.index).index(idx)
        exp = explainer.explain_instance(Xte[pos], core.predict, num_features=min(12, len(names)))
        for feat, weight in exp.as_list():
            lime_rows.append({"risk_case": label, "test_index": int(idx), "feature_rule": feat, "lime_weight": float(weight)})
        exp.as_pyplot_figure()
        plt.title(f"LIME — {label} risk")
        save_fig(f"lime_{label}_risk.png")
    lime_df = pd.DataFrame(lime_rows)
    save_table(lime_df, "lime_explanations.csv")
except Exception as e:
    log(f"LIME skipped: {repr(e)}")


In [ ]:
section("Counterfactual explanations: robust DiCE retries with fallback")

ACTIONABLE_FEATURES = [
    c for c in [
        "Sleep_Hours", "Physical_Activity_Hrs", "Social_Support_Score", "Financial_Stress", "Work_Stress",
        "Self_Esteem_Score", "Life_Satisfaction_Score", "Loneliness_Score", "Therapy", "Meditation"
    ] if c in SAFE_FEATURES
]

features_to_decrease = [c for c in ["Financial_Stress", "Work_Stress", "Loneliness_Score"] if c in SAFE_FEATURES]
features_to_increase = [c for c in ["Sleep_Hours", "Physical_Activity_Hrs", "Social_Support_Score", "Self_Esteem_Score", "Life_Satisfaction_Score", "Therapy", "Meditation"] if c in SAFE_FEATURES]

def impute_original_frame(X, ref):
    X = X.copy()
    for col in X.columns:
        if pd.api.types.is_numeric_dtype(ref[col]):
            fill = pd.to_numeric(ref[col], errors="coerce").median()
        else:
            mode = ref[col].dropna().mode()
            fill = mode.iloc[0] if len(mode) else "Missing"
        X[col] = X[col].fillna(fill)
    return X

def clip_to_training_range(X, ref):
    X = X.copy()
    for col in X.columns:
        if pd.api.types.is_numeric_dtype(ref[col]):
            lo = pd.to_numeric(ref[col], errors="coerce").quantile(0.01)
            hi = pd.to_numeric(ref[col], errors="coerce").quantile(0.99)
            if pd.notna(lo) and pd.notna(hi):
                X[col] = pd.to_numeric(X[col], errors="coerce").clip(lo, hi)
    return X

cf_df = pd.DataFrame()
dice_status = {"status": "not_run"}
try:
    import dice_ml
    train_imp = clip_to_training_range(impute_original_frame(X_train_safe, X_train_safe), X_train_safe)
    test_imp = clip_to_training_range(impute_original_frame(X_test_safe, X_train_safe), X_train_safe)
    dice_train = train_imp.copy()
    dice_train[TARGET] = y_train.values
    continuous = [c for c in SAFE_FEATURES if pd.api.types.is_numeric_dtype(train_imp[c])]
    data_dice = dice_ml.Data(dataframe=dice_train, continuous_features=continuous, outcome_name=TARGET)
    model_dice = dice_ml.Model(model=final_model, backend="sklearn", model_type="regressor")
    exp = dice_ml.Dice(data_dice, model_dice, method="random")

    pred_series = pd.Series(pred_test, index=X_test_safe.index).sort_values(ascending=False)
    query_candidates = list(pred_series.head(min(8, len(pred_series))).index)
    desired_ranges = [
        [float(y.min()), float(low_threshold)],
        [float(y.min()), float(y_train.quantile(0.45))],
        [float(y.min()), float(y_train.quantile(0.55))],
    ]
    dice_attempts = []
    for qid in query_candidates:
        query = test_imp.loc[[qid]].copy()
        query = clip_to_training_range(impute_original_frame(query, X_train_safe), X_train_safe)
        for desired_range in desired_ranges:
            for total_cfs in [4, 8, 12]:
                try:
                    dice_result = exp.generate_counterfactuals(
                        query,
                        total_CFs=total_cfs,
                        desired_range=desired_range,
                        features_to_vary=ACTIONABLE_FEATURES if ACTIONABLE_FEATURES else "all"
                    )
                    candidate = dice_result.cf_examples_list[0].final_cfs_df
                    if candidate is not None and len(candidate) > 0:
                        candidate = candidate.copy()
                        candidate["source_index"] = int(qid)
                        candidate["desired_low"] = desired_range[0]
                        candidate["desired_high"] = desired_range[1]
                        candidate["attempted_total_CFs"] = total_cfs
                        cf_df = candidate
                        dice_status = {"status": "completed", "source_index": int(qid), "desired_range": desired_range, "total_CFs": total_cfs}
                        raise StopIteration
                except StopIteration:
                    raise
                except Exception as e:
                    dice_attempts.append({"source_index": int(qid), "desired_range": desired_range, "total_CFs": total_cfs, "error": repr(e)})
    dice_status = {"status": "no_counterfactuals_found", "attempts": dice_attempts[:20]}
except StopIteration:
    pass
except Exception as e:
    dice_status = {"status": "failed", "reason": repr(e)}

if not cf_df.empty:
    save_table(cf_df, "dice_counterfactuals.csv")
    save_json(dice_status, "dice_status.json")
    display(cf_df)
else:
    log(f"DiCE did not return feasible counterfactuals; using expanded deterministic what-if analysis. Status: {dice_status}")
    high_idx = int(pd.Series(pred_test, index=X_test_safe.index).sort_values(ascending=False).index[0])
    base = clip_to_training_range(impute_original_frame(X_test_safe.loc[[high_idx]], X_train_safe), X_train_safe)
    base_pred = float(final_model.predict(base)[0])
    rows = []
    candidate_steps = []
    for feat in features_to_increase:
        candidate_steps += [(feat, 1), (feat, 2)]
    for feat in features_to_decrease:
        candidate_steps += [(feat, -1), (feat, -2)]
    for feat, delta in candidate_steps:
        if feat in base.columns:
            mod = base.copy()
            mod[feat] = mod[feat] + delta
            mod = clip_to_training_range(mod, X_train_safe)
            new_pred = float(final_model.predict(mod)[0])
            rows.append({"test_index": high_idx, "changed_feature": feat, "delta": delta, "base_prediction": base_pred, "new_prediction": new_pred, "predicted_reduction": base_pred-new_pred})
    # Pairwise combined interventions among the strongest single-feature interventions.
    temp = pd.DataFrame(rows).sort_values("predicted_reduction", ascending=False).head(5)
    for i in range(len(temp)):
        for j in range(i + 1, len(temp)):
            mod = base.copy()
            f1, d1 = temp.iloc[i]["changed_feature"], temp.iloc[i]["delta"]
            f2, d2 = temp.iloc[j]["changed_feature"], temp.iloc[j]["delta"]
            mod[f1] = mod[f1] + d1
            mod[f2] = mod[f2] + d2
            mod = clip_to_training_range(mod, X_train_safe)
            new_pred = float(final_model.predict(mod)[0])
            rows.append({"test_index": high_idx, "changed_feature": f"{f1}+{f2}", "delta": f"{d1},{d2}", "base_prediction": base_pred, "new_prediction": new_pred, "predicted_reduction": base_pred-new_pred})
    cf_df = pd.DataFrame(rows).sort_values("predicted_reduction", ascending=False)
    save_table(cf_df, "fallback_what_if_counterfactuals.csv")
    save_json(dice_status, "dice_status.json")
    display(cf_df.head(20))

if not cf_df.empty and "predicted_reduction" in cf_df.columns:
    plt.figure(figsize=(8,5))
    top_cf = cf_df.head(12).iloc[::-1]
    plt.barh(top_cf["changed_feature"].astype(str), top_cf["predicted_reduction"].astype(float))
    plt.xlabel("Predicted MHR reduction")
    plt.title("Best what-if intervention candidates")
    save_fig("counterfactual_what_if_reductions.png")


In [ ]:
section("Uncertainty-aware safety policy engine")

ensemble_preds = []
for seed in range(20):
    est = clone(final_model)
    for key in est.get_params().keys():
        if key.endswith("random_state"):
            est.set_params(**{key: RANDOM_STATE + seed})
    est.fit(X_train_safe, y_train)
    ensemble_preds.append(est.predict(X_test_safe))
ensemble_preds = np.vstack(ensemble_preds)
uncertainty = ensemble_preds.std(axis=0)
mean_pred = ensemble_preds.mean(axis=0)
confidence = 1 / (1 + uncertainty)
unc_threshold = float(np.quantile(uncertainty, 0.75))
conf_threshold = float(np.quantile(confidence, 0.25))

def risk_level(score):
    if score >= high_threshold:
        return "High Risk"
    if score >= low_threshold:
        return "Medium Risk"
    return "Low Risk"

def safety_action(score, unc, conf):
    rl = risk_level(score)
    if rl == "High Risk" and unc <= unc_threshold:
        return "Emergency escalation alert"
    if rl == "High Risk" and unc > unc_threshold:
        return "Urgent clinician review"
    if rl == "Medium Risk":
        return "Preventive intervention recommended"
    if rl == "Low Risk" and conf < conf_threshold:
        return "Precautionary monitoring escalation"
    return "Routine monitoring"

safety_df = pd.DataFrame({"Actual_MHR": y_test.values, "Predicted_MHR": pred_test, "Ensemble_Mean": mean_pred, "Uncertainty": uncertainty, "Confidence": confidence})
safety_df["Risk_Level"] = safety_df["Predicted_MHR"].apply(risk_level)
safety_df["Safety_Action"] = [safety_action(s,u,c) for s,u,c in zip(safety_df["Predicted_MHR"], safety_df["Uncertainty"], safety_df["Confidence"])]
safety_df["Escalation_Flag"] = safety_df["Safety_Action"].isin(["Emergency escalation alert", "Urgent clinician review"]).astype(int)
save_table(safety_df, "uncertainty_aware_safety_policy_outputs.csv")
display(safety_df.head())

plt.figure(figsize=(7,4))
plt.scatter(safety_df["Predicted_MHR"], safety_df["Uncertainty"], alpha=0.75)
plt.axhline(unc_threshold, linestyle="--")
plt.xlabel("Predicted MHR")
plt.ylabel("Uncertainty")
plt.title("Uncertainty across risk continuum")
save_fig("uncertainty_vs_risk.png")

plt.figure(figsize=(8,4))
safety_df["Safety_Action"].value_counts().plot(kind="bar")
plt.ylabel("Count")
plt.title("Safety action distribution")
save_fig("safety_action_distribution.png")


In [ ]:
section("Subgroup diagnostics, missing-data robustness, and target-weight sensitivity")

subgroup_cols = [c for c in ["Gender", "Education_Level", "Employment_Status", "Medication_Use", "Therapy", "Meditation", "Substance_Use"] if c in X_test_safe.columns]
sub_rows = []
for col in subgroup_cols:
    tmp = pd.DataFrame({"group": X_test_safe[col].astype(str), "y": y_test.values, "pred": pred_test})
    for group, g in tmp.groupby("group"):
        if len(g) >= 5:
            sub_rows.append({"feature": col, "group": group, "n": int(len(g)), "mae": mean_absolute_error(g["y"], g["pred"]), "rmse": rmse(g["y"], g["pred"]), "mean_error": float(np.mean(g["y"]-g["pred"]))})
subgroup_df = pd.DataFrame(sub_rows).sort_values(["feature", "rmse"]) if sub_rows else pd.DataFrame()
save_table(subgroup_df, "subgroup_error_diagnostics.csv")
display(subgroup_df)

rng = check_random_state(RANDOM_STATE)
miss_rows = []
for frac in [0, 0.05, 0.10, 0.20, 0.30]:
    Xc = X_test_safe.copy()
    if frac > 0:
        Xc = Xc.mask(rng.rand(*Xc.shape) < frac)
    pc = final_model.predict(Xc)
    row = {"missing_fraction": frac}
    row.update(metrics_dict(y_test, pc, "test_"))
    miss_rows.append(row)
missing_df = pd.DataFrame(miss_rows)
save_table(missing_df, "missing_data_robustness.csv")
display(missing_df)

weight_sets = {
    "baseline_40_30_30": {"Depression_Score": 0.40, "Anxiety_Score": 0.30, "Stress_Level": 0.30},
    "equal_33_33_33": {"Depression_Score": 1/3, "Anxiety_Score": 1/3, "Stress_Level": 1/3},
    "depression_heavy_50_25_25": {"Depression_Score": 0.50, "Anxiety_Score": 0.25, "Stress_Level": 0.25},
    "anxiety_heavy_25_50_25": {"Depression_Score": 0.25, "Anxiety_Score": 0.50, "Stress_Level": 0.25},
    "stress_heavy_25_25_50": {"Depression_Score": 0.25, "Anxiety_Score": 0.25, "Stress_Level": 0.50},
}
sens = []
for name, w in weight_sets.items():
    y_alt = sum(w[c] * pd.to_numeric(df[c], errors="coerce") for c in w)
    y_alt_train, y_alt_test = y_alt.loc[X_train_safe.index], y_alt.loc[X_test_safe.index]
    m = Pipeline([("pre", clone(pre_safe)), ("model", GradientBoostingRegressor(random_state=RANDOM_STATE))])
    m.fit(X_train_safe, y_alt_train)
    p = m.predict(X_test_safe)
    row = {"weight_setting": name, "weights": json.dumps(w), "target_correlation_with_baseline": float(np.corrcoef(y_alt, y)[0,1])}
    row.update(metrics_dict(y_alt_test, p, "test_"))
    sens.append(row)
sens_df = pd.DataFrame(sens).sort_values("test_rmse")
save_table(sens_df, "target_weight_sensitivity.csv")
display(sens_df)


In [ ]:
section("Automatic report generation")

report_md = REPORT_DIR / "MHR_SafeGuard_research_report.md"
with open(report_md, "w", encoding="utf-8") as f:
    f.write("# MHR-SafeGuard Research Report\n\n")
    f.write(f"Generated: {now()}\n\n")
    f.write(f"Dataset: `{dataset_path}`\n\n")
    f.write(f"Shape: {df_raw.shape}\n\n")
    f.write("Target: MHR = 0.40 Depression + 0.30 Anxiety + 0.30 Stress.\n\n")
    f.write("Preferred setting: leakage-aware features excluding Depression, Anxiety, and Stress.\n\n")
    f.write("## Final model\n\n")
    for k, v in final_metrics.items():
        f.write(f"- {k}: {v}\n")
    f.write("\n## Conformal prediction\n\n")
    for k, v in conformal_summary.items():
        f.write(f"- {k}: {v}\n")
    f.write("\n## Calibration\n\n")
    for k, v in cal_summary.items():
        f.write(f"- {k}: {v}\n")
    f.write(f"\nOutputs: `{BASE_DIR}`\n")
log(f"Saved markdown report: {report_md}")

try:
    from docx import Document
    doc = Document()
    doc.add_heading("MHR-SafeGuard Research Report", 0)
    doc.add_paragraph(f"Generated: {now()}")
    doc.add_heading("Dataset", level=1)
    doc.add_paragraph(str(dataset_path))
    doc.add_paragraph(f"Shape: {df_raw.shape}")
    doc.add_heading("Final model", level=1)
    for k, v in final_metrics.items():
        doc.add_paragraph(f"{k}: {v}")
    doc.add_heading("Conformal prediction", level=1)
    for k, v in conformal_summary.items():
        doc.add_paragraph(f"{k}: {v}")
    doc.add_heading("Calibration", level=1)
    for k, v in cal_summary.items():
        doc.add_paragraph(f"{k}: {v}")
    doc.add_paragraph(f"Outputs: {BASE_DIR}")
    docx_path = REPORT_DIR / "MHR_SafeGuard_research_report.docx"
    doc.save(docx_path)
    log(f"Saved Word report: {docx_path}")
except Exception as e:
    log(f"Word report skipped: {repr(e)}")


In [ ]:
section("Manuscript-ready exports and HTML dashboard")

# Export selected tables as LaTeX for manuscript integration.
latex_tables = {
    "model_comparison_valid.tex": valid_results.head(12),
    "final_metrics.tex": pd.DataFrame([final_metrics]),
    "bootstrap_ci.tex": ci_df,
    "conformal_summary.tex": pd.DataFrame([conformal_summary]),
    "calibration_summary.tex": pd.DataFrame([cal_summary]),
}
if "rep_summary" in globals() and isinstance(rep_summary, pd.DataFrame) and not rep_summary.empty:
    latex_tables["repeated_cv_summary.tex"] = rep_summary
if "stats_df" in globals() and isinstance(stats_df, pd.DataFrame) and not stats_df.empty:
    latex_tables["repeated_cv_statistical_tests.tex"] = stats_df

for name, table in latex_tables.items():
    try:
        path = TABLE_DIR / name
        path.write_text(table.to_latex(index=False, escape=True), encoding="utf-8")
        log(f"Saved LaTeX table: {path}")
    except Exception as e:
        log(f"LaTeX export skipped for {name}: {repr(e)}")

# A lightweight HTML dashboard using static saved figures and concise metrics.
html_path = REPORT_DIR / "MHR_SafeGuard_dashboard.html"
figures = sorted(FIG_DIR.glob("*.png"))
figure_html = "\n".join([f'<h3>{p.name}</h3><img src="../Figures/{p.name}" style="max-width: 900px; width: 100%; border: 1px solid #ddd; margin-bottom: 20px;">' for p in figures])
metric_items = "\n".join([f"<li><b>{k}</b>: {v}</li>" for k, v in final_metrics.items()])
html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>MHR-SafeGuard Research Dashboard</title>
<style>
body {{ font-family: Arial, sans-serif; margin: 32px; line-height: 1.45; }}
h1, h2 {{ color: #1f2937; }}
.code {{ background: #f6f8fa; padding: 10px; border-radius: 6px; }}
img {{ display: block; }}
</style>
</head>
<body>
<h1>MHR-SafeGuard Research Dashboard</h1>
<p><b>Generated:</b> {now()}</p>
<p><b>Dataset:</b> <span class="code">{dataset_path}</span></p>
<p><b>Output directory:</b> <span class="code">{BASE_DIR}</span></p>
<h2>Final Metrics</h2>
<ul>{metric_items}</ul>
<h2>Optuna Refinement</h2>
<pre>{json.dumps(optuna_summary, indent=2)}</pre>
<h2>Conformal Prediction</h2>
<pre>{json.dumps(conformal_summary, indent=2)}</pre>
<h2>Calibration</h2>
<pre>{json.dumps(cal_summary, indent=2)}</pre>
<h2>Figures</h2>
{figure_html}
</body>
</html>
"""
html_path.write_text(html, encoding="utf-8")
log(f"Saved HTML dashboard: {html_path}")


# Long-term framework additions

The following sections convert the single-paper notebook into a reusable research pipeline with configuration artifacts, optional tracking, external-validation hooks, distribution-shift diagnostics, model/data cards, and inference assets.

In [ ]:
section("Experiment configuration and run manifest")

EXPERIMENT_CONFIG = {
    "project_name": PROJECT_NAME,
    "dataset_path": str(dataset_path),
    "target": TARGET,
    "target_formula": target_summary.get("formula", ""),
    "feature_modes": {"full_features": ALL_FEATURES, "leakage_aware_features": SAFE_FEATURES},
    "excluded_leakage_components": LEAKAGE_COMPONENTS,
    "random_state": RANDOM_STATE,
    "output_directory": str(BASE_DIR),
    "model_artifact": str(model_path),
    "recommended_feature_mode": "leakage_aware_features",
    "notes": [
        "The leakage-aware setting excludes direct target components from predictors.",
        "The full-feature setting is retained as a leakage diagnostic baseline.",
        "Clinical use requires external validation, clinician review, and prospective evaluation.",
    ],
}
config_json = BASE_DIR / "experiment_config.json"
config_json.write_text(json.dumps(EXPERIMENT_CONFIG, indent=2), encoding="utf-8")
log(f"Saved experiment configuration: {config_json}")
config_yaml = BASE_DIR / "experiment_config.yaml"
with open(config_yaml, "w", encoding="utf-8") as f:
    for key, value in EXPERIMENT_CONFIG.items():
        f.write(f"{key}: {json.dumps(value, ensure_ascii=False)}\n")
log(f"Saved YAML-style configuration: {config_yaml}")
RUN_MANIFEST = {
    "generated_at": now(),
    "base_dir": str(BASE_DIR),
    "figures": [p.name for p in sorted(FIG_DIR.glob("*.png"))],
    "tables": [p.name for p in sorted(TABLE_DIR.glob("*"))],
    "models": [p.name for p in sorted(MODEL_DIR.glob("*"))],
    "reports": [p.name for p in sorted(REPORT_DIR.glob("*"))],
}
save_json(RUN_MANIFEST, "run_manifest.json")
print(json.dumps(EXPERIMENT_CONFIG, indent=2)[:3000])

In [ ]:
section("Optional experiment tracking")

tracking_status = {"enabled": False, "backend": "none", "reason": "mlflow not installed"}
try:
    import mlflow
    tracking_dir = BASE_DIR / "MLflow"
    tracking_dir.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(tracking_dir.as_uri())
    mlflow.set_experiment(PROJECT_NAME)
    with mlflow.start_run(run_name=f"{PROJECT_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
        mlflow.log_params({"target": TARGET, "best_model": BEST_MODEL_NAME, "feature_mode": "leakage_aware_features", "random_state": RANDOM_STATE})
        for k, v in final_metrics.items():
            if isinstance(v, (int, float, np.integer, np.floating)):
                mlflow.log_metric(k, float(v))
        for artifact in [model_path, LOG_PATH, BASE_DIR / "experiment_config.json"]:
            if Path(artifact).exists():
                mlflow.log_artifact(str(artifact))
    tracking_status = {"enabled": True, "backend": "mlflow", "tracking_uri": tracking_dir.as_uri()}
    log("MLflow tracking completed")
except Exception as e:
    tracking_status = {"enabled": False, "backend": "none", "reason": repr(e)}
    log(f"Experiment tracking skipped: {repr(e)}")
save_json(tracking_status, "experiment_tracking_status.json")

In [ ]:
section("Distribution-shift and external-validation diagnostics")

def _safe_numeric_array(s):
    return pd.to_numeric(s, errors="coerce").dropna().astype(float).values

def population_stability_index(expected, actual, bins=10, eps=1e-6):
    expected = _safe_numeric_array(pd.Series(expected)); actual = _safe_numeric_array(pd.Series(actual))
    if len(expected) < 2 or len(actual) < 2: return np.nan
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        edges = np.linspace(min(expected.min(), actual.min()), max(expected.max(), actual.max()), bins + 1)
    edges[0] = -np.inf; edges[-1] = np.inf
    e_counts, _ = np.histogram(expected, bins=edges); a_counts, _ = np.histogram(actual, bins=edges)
    e_pct = np.clip(e_counts / max(e_counts.sum(), 1), eps, None)
    a_pct = np.clip(a_counts / max(a_counts.sum(), 1), eps, None)
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))

def categorical_psi(expected, actual, eps=1e-6):
    e = pd.Series(expected).astype(str).fillna("<missing>").value_counts(normalize=True)
    a = pd.Series(actual).astype(str).fillna("<missing>").value_counts(normalize=True)
    cats = sorted(set(e.index) | set(a.index))
    ev = np.clip(np.array([e.get(c, 0.0) for c in cats]), eps, None)
    av = np.clip(np.array([a.get(c, 0.0) for c in cats]), eps, None)
    return float(np.sum((av - ev) * np.log(av / ev)))

def rbf_mmd2(X, Y, max_samples=400, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state); X = np.asarray(X, dtype=float); Y = np.asarray(Y, dtype=float)
    if X.shape[0] > max_samples: X = X[rng.choice(X.shape[0], max_samples, replace=False)]
    if Y.shape[0] > max_samples: Y = Y[rng.choice(Y.shape[0], max_samples, replace=False)]
    if X.size == 0 or Y.size == 0: return np.nan
    Z = np.vstack([X, Y]); n = min(250, Z.shape[0]); Zs = Z[rng.choice(Z.shape[0], n, replace=False)] if Z.shape[0] > n else Z
    d2 = ((Zs[:, None, :] - Zs[None, :, :]) ** 2).sum(axis=2)
    med = np.median(d2[d2 > 0]) if np.any(d2 > 0) else 1.0
    gamma = 1.0 / (2.0 * med) if med > 0 else 1.0
    Kxx = np.exp(-gamma * ((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)).mean()
    Kyy = np.exp(-gamma * ((Y[:, None, :] - Y[None, :, :]) ** 2).sum(axis=2)).mean()
    Kxy = np.exp(-gamma * ((X[:, None, :] - Y[None, :, :]) ** 2).sum(axis=2)).mean()
    return float(Kxx + Kyy - 2 * Kxy)

def compute_shift_report(reference_df, comparison_df, features, label):
    rows = []
    for c in features:
        if c not in reference_df.columns or c not in comparison_df.columns: continue
        if pd.api.types.is_numeric_dtype(reference_df[c]) and pd.api.types.is_numeric_dtype(comparison_df[c]):
            ref = _safe_numeric_array(reference_df[c]); cmp = _safe_numeric_array(comparison_df[c])
            if len(ref) > 1 and len(cmp) > 1:
                ks_stat, ks_p = stats.ks_2samp(ref, cmp); psi = population_stability_index(ref, cmp); ed = float(stats.energy_distance(ref, cmp))
            else:
                ks_stat, ks_p, psi, ed = np.nan, np.nan, np.nan, np.nan
            rows.append({"comparison": label, "feature": c, "type": "numeric", "ks_stat": ks_stat, "ks_p": ks_p, "psi": psi, "energy_distance": ed})
        else:
            rows.append({"comparison": label, "feature": c, "type": "categorical", "ks_stat": np.nan, "ks_p": np.nan, "psi": categorical_psi(reference_df[c], comparison_df[c]), "energy_distance": np.nan})
    return pd.DataFrame(rows)

shift_train_test = compute_shift_report(X_train_safe, X_test_safe, SAFE_FEATURES, "holdout_test_vs_train")
save_table(shift_train_test, "distribution_shift_train_test_extended.csv")
display(shift_train_test.sort_values("psi", ascending=False).head(15))
try:
    Xtr_trans = final_model.named_steps["pre"].transform(X_train_safe); Xte_trans = final_model.named_steps["pre"].transform(X_test_safe)
    mmd_summary = {"comparison": "holdout_test_vs_train", "rbf_mmd2": rbf_mmd2(Xtr_trans, Xte_trans)}
except Exception as e:
    mmd_summary = {"comparison": "holdout_test_vs_train", "rbf_mmd2": None, "error": repr(e)}
save_json(mmd_summary, "distribution_shift_mmd_summary.json")

EXTERNAL_DATA_PATH = ROOT / "Datasets" / "MentalHealth" / "external_anxiety_depression_data.csv"
external_status = {"external_validation_enabled": False, "expected_path": str(EXTERNAL_DATA_PATH)}
if EXTERNAL_DATA_PATH.exists():
    ext_raw = pd.read_csv(EXTERNAL_DATA_PATH); ext = ext_raw.copy()
    if all(c in ext.columns for c in WEIGHTS):
        ext["MHR"] = sum(WEIGHTS[c] * pd.to_numeric(ext[c], errors="coerce") for c in WEIGHTS)
        ext_X = ext.reindex(columns=SAFE_FEATURES); ext_y = ext["MHR"]; ext_pred = final_model.predict(ext_X)
        external_status.update({"external_validation_enabled": True, "rows": int(len(ext)), **metrics_dict(ext_y, ext_pred, "external_")})
        ext_pred_df = ext_X.copy(); ext_pred_df["Actual_MHR"] = ext_y.values; ext_pred_df["Predicted_MHR"] = ext_pred; ext_pred_df["Residual"] = ext_y.values - ext_pred
        save_table(ext_pred_df, "external_validation_predictions.csv")
        save_table(compute_shift_report(X_train_safe, ext_X, SAFE_FEATURES, "external_vs_train"), "distribution_shift_external_extended.csv")
    else:
        external_status.update({"reason": "External file is missing target-construction columns."})
else:
    template_path = BASE_DIR / "external_validation_template.csv"; df_raw.head(3).to_csv(template_path, index=False); external_status.update({"template_created": str(template_path)})
save_json(external_status, "external_validation_status.json")
print(json.dumps({"mmd": mmd_summary, "external": external_status}, indent=2))

In [ ]:
section("Model card, data card, and inference assets")

model_card_path = REPORT_DIR / "MODEL_CARD_MHR_SafeGuard.md"
data_card_path = REPORT_DIR / "DATA_CARD_MentalHealthFactors.md"

model_card = f'''# Model Card — MHR-SafeGuard

## Intended use
This model supports research on continuous mental-health risk estimation from structured behavioral and lifestyle variables. It is not a standalone diagnostic or emergency triage tool.

## Model
- Best model: `{BEST_MODEL_NAME}`
- Artifact: `{model_path}`
- Feature mode: leakage-aware features
- Target: `{target_summary.get('formula', '')}`

## Performance on hold-out set
- MAE: {final_metrics.get('test_mae')}
- RMSE: {final_metrics.get('test_rmse')}
- R2: {final_metrics.get('test_r2')}

## Safeguards and limitations
- Direct target components are excluded from the preferred predictor set.
- Predictions should be interpreted with uncertainty, conformal intervals, and safety-policy outputs.
- External validation, clinician review, and prospective evaluation are required before clinical use.
- Counterfactual outputs are algorithmic what-if scenarios, not medical prescriptions.

Generated: {now()}
'''
model_card_path.write_text(model_card, encoding="utf-8")
log(f"Saved model card: {model_card_path}")

data_card = f'''# Data Card — Anxiety and Depression Mental Health Factors

## Source
Dataset used from: `{dataset_path}`

## Shape
- Rows: {df_raw.shape[0]}
- Columns: {df_raw.shape[1]}

## Target construction
`MHR = 0.40*Depression_Score + 0.30*Anxiety_Score + 0.30*Stress_Level`

## Preferred predictors
The leakage-aware predictor set excludes direct target components:
`{', '.join(LEAKAGE_COMPONENTS)}`

## Known cautions
The dataset should not be interpreted as a substitute for clinical assessment. External validation, fairness analysis, and domain review are needed for deployment.

Generated: {now()}
'''
data_card_path.write_text(data_card, encoding="utf-8")
log(f"Saved data card: {data_card_path}")

inference_script = MODEL_DIR / "mhr_safeguard_batch_inference.py"
inference_code = '''from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd


def load_artifacts(model_path):
    model_path = Path(model_path)
    model = joblib.load(model_path)
    config_path = model_path.parents[1] / "experiment_config.json"
    config = json.loads(config_path.read_text(encoding="utf-8")) if config_path.exists() else {}
    return model, config


def predict_file(input_csv, output_csv, model_path):
    model, config = load_artifacts(model_path)
    df = pd.read_csv(input_csv)
    features = config.get("feature_modes", {}).get("leakage_aware_features")
    X = df.reindex(columns=features) if features else df
    pred = model.predict(X)
    out = df.copy()
    out["Predicted_MHR"] = pred
    out["Risk_Level"] = pd.cut(pred, bins=[-np.inf, np.quantile(pred, 0.33), np.quantile(pred, 0.66), np.inf], labels=["Low", "Medium", "High"]).astype(str)
    out.to_csv(output_csv, index=False)
    return output_csv


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description="Batch inference for MHR-SafeGuard.")
    parser.add_argument("--input_csv", required=True)
    parser.add_argument("--output_csv", required=True)
    parser.add_argument("--model_path", required=True)
    args = parser.parse_args()
    predict_file(args.input_csv, args.output_csv, args.model_path)
'''
inference_script.write_text(inference_code, encoding="utf-8")
log(f"Saved batch inference script: {inference_script}")

inference_template = BASE_DIR / "new_patient_input_template.csv"
df_raw.reindex(columns=SAFE_FEATURES).head(5).to_csv(inference_template, index=False)
log(f"Saved inference input template: {inference_template}")

readme_path = BASE_DIR / "README_REPRODUCIBILITY.md"
readme_path.write_text(f'''# MHR-SafeGuard Reproducibility Package

## Main notebook
`MHR_SafeGuard_Polished_Research_Colab_v3_LongTerm_Framework.ipynb`

## Main outputs
- Progressive log: `{LOG_PATH.name}`
- Configuration: `experiment_config.json`
- Model: `Models/{model_path.name}`
- Reports: `Reports/`
- Tables: `Tables/`
- Figures: `Figures/`

## Batch inference example
```bash
python Models/mhr_safeguard_batch_inference.py \
  --input_csv new_patient_input_template.csv \
  --output_csv new_patient_predictions.csv \
  --model_path Models/{model_path.name}
```

Generated: {now()}
''', encoding="utf-8")
log(f"Saved reproducibility README: {readme_path}")

save_json({"model_card": str(model_card_path), "data_card": str(data_card_path), "inference_script": str(inference_script), "inference_template": str(inference_template), "readme": str(readme_path)}, "documentation_and_inference_assets.json")

In [ ]:
section("Long-term framework summary")
long_term_summary = {
    "configuration": str(BASE_DIR / "experiment_config.json"),
    "tracking_status": str(TABLE_DIR / "experiment_tracking_status.json"),
    "external_validation_status": str(TABLE_DIR / "external_validation_status.json"),
    "shift_train_test": str(TABLE_DIR / "distribution_shift_train_test_extended.csv"),
    "model_card": str(REPORT_DIR / "MODEL_CARD_MHR_SafeGuard.md"),
    "data_card": str(REPORT_DIR / "DATA_CARD_MentalHealthFactors.md"),
    "inference_script": str(MODEL_DIR / "mhr_safeguard_batch_inference.py"),
    "inference_template": str(BASE_DIR / "new_patient_input_template.csv"),
}
save_json(long_term_summary, "long_term_framework_summary.json")
print(json.dumps(long_term_summary, indent=2))

In [ ]:
section("Final execution summary")

print("\nAll outputs saved in:", BASE_DIR)
print("Progressive log saved in:", LOG_PATH)
print("Final model:", BEST_MODEL_NAME)
print("Final test RMSE:", final_metrics.get("test_rmse"))
print("Final test R2:", final_metrics.get("test_r2"))

print("Configuration:", BASE_DIR / "experiment_config.json")
print("Model card:", REPORT_DIR / "MODEL_CARD_MHR_SafeGuard.md")
print("Data card:", REPORT_DIR / "DATA_CARD_MentalHealthFactors.md")
print("Inference template:", BASE_DIR / "new_patient_input_template.csv")
log("Notebook completed successfully")
